In [ ]:
import xarray as xr
import numpy as np

fp1a = 'data/chlorophyll/cmems_mod_glo_bgc_my_0.25deg_P1M-m_1782417984886.nc'
ds1a = xr.open_dataset(fp1a)

with xr.set_options(keep_attrs=True):
    chl_depth_mean = ds1a["chl"].mean(dim="depth")

ds_processed = chl_depth_mean.to_dataset(name="chl")
ds_processed.attrs = ds1a.attrs

# --- Integer packing: the main size win ---
valid_min, valid_max = 0.0, 20.0  # chl >= 0; use attrs' valid_max as ceiling
scale_factor = (valid_max - valid_min) / (2**16 - 2)
add_offset = valid_min

encoding = {
    "chl": {
        "dtype": "int16",
        "scale_factor": np.float32(scale_factor),
        "add_offset": np.float32(add_offset),
        "_FillValue": -32768,
        "zlib": True,
        "complevel": 9,
        "shuffle": True,
    }
}

ds_processed.to_netcdf("data/chlorophyll/chl_depth_mean.nc", encoding=encoding)